# FlowGuard - Phase 2: Contrastive Pre-training

In [ ]:
import os, sys

REPO_DIR = "/content/flowguard"
if not os.path.exists(REPO_DIR):
    raise RuntimeError("Run notebook 00_setup_and_validate.ipynb first.")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from src.utils.config import load_config, get_device
from src.utils.reproducibility import setup_reproducibility

config = load_config("configs/phase2_contrastive.yaml")
setup_reproducibility(config)
device = get_device(config)
print(f"Device: {device}")

In [ ]:
import os
assert os.path.exists("data/processed/combined_unlabeled.parquet"), "Run preprocessing first!"
print(f"Unlabeled data: {os.path.getsize('data/processed/combined_unlabeled.parquet') / 1e6:.0f} MB")

## Train Contrastive Encoder

In [ ]:
from src.training.contrastive import train_contrastive
history = train_contrastive("configs/phase2_contrastive.yaml")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(history['loss'])
plt.xlabel('Epoch')
plt.ylabel('NT-Xent Loss')
plt.title('Contrastive Pre-training Loss')
plt.savefig('results/plots/phase2_loss.png', dpi=100)
plt.show()

## UMAP Visualization

In [ ]:
import torch
from src.models.transformer_encoder import FlowTransformerEncoder
from src.data.dataset import create_dataloader
from src.evaluation.visualization import extract_embeddings, plot_umap

enc_cfg = config['model']['encoder']
encoder = FlowTransformerEncoder(
    input_dim=enc_cfg.get('input_dim', 49),
    model_dim=enc_cfg.get('model_dim', 128),
).to(device)
encoder.load_state_dict(torch.load("checkpoints/phase2/best_encoder.pt", map_location=device))

# Visualize embeddings from each dataset
import numpy as np
all_emb, all_labels, all_ds = [], [], []
for i, ds_info in enumerate(config['data']['datasets']):
    name = ds_info['name']
    path = f"data/splits/protocol_a/{name}_test.parquet"
    if not os.path.exists(path):
        continue
    loader = create_dataloader(path, batch_size=1024, shuffle=False)
    emb, labels = extract_embeddings(encoder, loader, device, max_samples=2000)
    all_emb.append(emb)
    all_labels.append(labels)
    all_ds.extend([name] * len(emb))

all_emb = np.concatenate(all_emb)
all_labels = np.concatenate(all_labels)
plot_umap(all_emb, all_labels, label_names={0: 'Benign', 1: 'Attack'},
          title="Phase 2 Encoder Embeddings", save_path="results/plots/phase2_umap.png")